# 202 · Checkpoint Evolution Analysis (S5-4K, 55M)

**Question**: Do improvements across training steps reflect better learning (beta + decay convergence) or just more parameters?

Since all 4 checkpoints share the same architecture (j2504167, 55M params, 4K context), this analysis isolates **training quality** from **model capacity**.

**Checkpoints**: step 22820, 38335, 79688, 100378 — each with 60 experiments × 2048 samples.

In [1]:
import numpy as np
import pandas as pd
import re, gc, math, json
from pathlib import Path
from collections import OrderedDict
from scipy.stats import linregress
from scipy.optimize import curve_fit
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

/homes/80/georgenigm/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -- Publication figure style --
SINGLE_W = 520
FULL_W   = 1080
IMG_SCALE = 3

_AX = dict(
    showline=True, linewidth=1.5, linecolor='black', mirror=True,
    showgrid=True, gridwidth=0.5, gridcolor='rgba(0,0,0,0.08)',
    ticks='outside', tickwidth=1, ticklen=4, tickcolor='black',
    zeroline=False,
)

def pub_layout(fig, width=SINGLE_W, height=None, legend_pos='tr', **kw):
    if height is None:
        height = int(width * 0.75)
    leg = {
        'tr': dict(x=0.98, y=0.98, xanchor='right', yanchor='top'),
        'br': dict(x=0.98, y=0.02, xanchor='right', yanchor='bottom'),
        'tl': dict(x=0.02, y=0.98, xanchor='left',  yanchor='top'),
        'bl': dict(x=0.02, y=0.02, xanchor='left',  yanchor='bottom'),
        'tc': dict(x=0.3, y=0.98, xanchor='center', yanchor='top'),
        'none': dict(visible=False),
    }.get(legend_pos, {})
    fig.update_layout(
        width=width, height=height,
        template='plotly_white',
        font=dict(family='Times New Roman, DejaVu Serif, serif', size=13, color='black'),
        title=None,
        margin=dict(l=60, r=15, t=15, b=55),
        legend=dict(**leg, bgcolor='rgba(255,255,255,0.85)',
                    bordercolor='black', borderwidth=1, font_size=11),
        **kw,
    )
    fig.update_xaxes(**_AX)
    fig.update_yaxes(**_AX)
    return fig

def save_fig(fig, name):
    fig.write_image(str(FIG_DIR / name), scale=IMG_SCALE)
    print(f"  Saved: {name}")

print("Publication style loaded.")

Publication style loaded.


In [ ]:
# -- Constants & checkpoint configuration --
TICK_SIZE = 100
MAX_SAMPLES = 500
MIDPRICE_MAX = 2_000_000
N_BOOTSTRAP = 1000
N_COND_MSGS = 500

# 4 checkpoints of j2504167 (55M, 4K context, 24tok encoding)
CHECKPOINTS = OrderedDict([
    ('Step 22820',  {'step': 22820,  'key': 'aggressive_scenario_v3/j2504167_step22820'}),
    ('Step 38335',  {'step': 38335,  'key': 'aggressive_scenario_v3/j2504167_step38335'}),
    ('Step 79688',  {'step': 79688,  'key': 'aggressive_scenario_v3/j2504167_step79688'}),
    ('Step 100378', {'step': 100378, 'key': 'aggressive_scenario_v3/j2504167_step100378'}),
])

STEPS = [cfg['step'] for cfg in CHECKPOINTS.values()]

# Sequential blue palette: light -> dark = early -> late training
STEP_COLORS = OrderedDict([
    ('Step 22820',  '#a6cee3'),
    ('Step 38335',  '#1f78b4'),
    ('Step 79688',  '#33a02c'),
    ('Step 100378', '#e31a1c'),
])

_BASE = [Path("/app/output/evalsequences"),
         Path("/scratch/local/homes/80/georgenigm/LOBS5/output/evalsequences")]
EVAL_BASE = next((p for p in _BASE if p.exists()), _BASE[-1])

_SDM = [Path("/app/lob_impact/sample_day_map.csv"),
        Path("/scratch/local/homes/80/georgenigm/LOBS5/lob_impact/sample_day_map.csv")]
SDM_PATH = next((p for p in _SDM if p.exists()), _SDM[-1])
SAMPLE_DAY_MAP = pd.read_csv(SDM_PATH)

_FIG = [Path("/app/pics_for_202_checkpoint_evolution"),
        Path("/homes/80/georgenigm/LOBS5/pics_for_202_checkpoint_evolution")]
FIG_DIR = next((p for p in _FIG if p.exists() or p.parent.exists()), _FIG[0])
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"EVAL_BASE : {EVAL_BASE}")
print(f"SDM       : {len(SAMPLE_DAY_MAP)} rows")
print(f"FIG_DIR   : {FIG_DIR}")
print(f"Checkpoints: {list(CHECKPOINTS.keys())}")

EVAL_BASE : /scratch/local/homes/80/georgenigm/LOBS5/output/evalsequences
SDM       : 2048 rows
FIG_DIR   : /homes/80/georgenigm/LOBS5/pics_for_202_checkpoint_evolution
Checkpoints: ['Step 22820', 'Step 38335', 'Step 79688', 'Step 100378']


In [4]:
# ── Data I/O helpers ──────────────────────────────────────────────

def discover_v2_folders(buy_path, sell_path):
    pattern = re.compile(r'^i(\d+)_c(\d+)_mb(\d+)_v(\d+)_cntxt(.+)$')
    rows = []
    for p in sorted(buy_path.iterdir()):
        if not p.is_dir():
            continue
        m = pattern.match(p.name)
        if not m:
            continue
        i, c, mb, V = int(m.group(1)), int(m.group(2)), int(m.group(3)), int(m.group(4))
        sell_p = sell_path / p.name
        if not sell_p.exists():
            continue
        rows.append({'folder': p.name, 'i': i, 'c': c, 'mb': mb, 'V': V,
                     'Q_total': i * V, 'buy_path': p, 'sell_path': sell_p})
    return pd.DataFrame(rows)


def parse_folder_params_v2(folder_name):
    m = re.match(r'i(\d+)_c(\d+)_mb(\d+)_v(\d+)_cntxt(.+)', folder_name)
    if m:
        return int(m.group(1)), int(m.group(2)), int(m.group(3)), int(m.group(4))
    return None, None, None, None


def compute_midprice(book_array):
    return (book_array[:, 0] + book_array[:, 2]) / 2


def compute_midprice_returns(books, min_len):
    returns = []
    for sid, book_array in books.items():
        midprice = compute_midprice(book_array[:min_len])
        returns.append(midprice - midprice[0])
    return np.stack(returns, axis=0)


def is_midprice_outlier(book_array, max_mp):
    mp = compute_midprice(book_array)
    return np.any(mp > max_mp) or np.any(mp <= 0)


def load_aggressive_indices(data_path):
    f = data_path / 'aggressive_indices.csv'
    if not f.exists():
        return np.array([], dtype=int)
    idx = np.loadtxt(f, dtype=int)
    return np.atleast_1d(idx)


def discover_data_params(data_path, max_samples=None):
    cond_dir = data_path / "data_cond"
    pat = re.compile(r"^(.+?)_(\d{4}-\d{2}-\d{2})_orderbook_real_id_(\d+)\.csv$")
    samples = []
    for f in cond_dir.glob("*_orderbook_real_id_*.csv"):
        m = pat.match(f.name)
        if m:
            samples.append((m.group(1), m.group(2), int(m.group(3))))
    samples.sort()
    if max_samples and len(samples) > max_samples:
        rng = np.random.RandomState(42)
        idx = rng.choice(len(samples), size=max_samples, replace=False)
        samples = [samples[i] for i in sorted(idx)]
    return samples


def load_folder_data(data_path, max_samples=None, max_midprice=None):
    samples = discover_data_params(data_path, max_samples)
    gen_books, gen_msgs, cond_lens = {}, {}, {}
    for ticker, date, sid in samples:
        cond_bp = data_path / f"data_cond/{ticker}_{date}_orderbook_real_id_{sid}.csv"
        gen_bp  = data_path / f"data_gen/{ticker}_{date}_orderbook_real_id_{sid}_gen_id_0.csv"
        gen_mp  = data_path / f"data_gen/{ticker}_{date}_message_real_id_{sid}_gen_id_0.csv"
        if not gen_bp.exists():
            continue
        cond_book = np.loadtxt(cond_bp, delimiter=',')
        gen_book  = np.loadtxt(gen_bp, delimiter=',')
        full_book = np.vstack([cond_book, gen_book])
        if max_midprice and is_midprice_outlier(full_book, max_midprice):
            continue
        gen_msg  = np.loadtxt(gen_mp, delimiter=',')
        cond_mp  = data_path / f"data_cond/{ticker}_{date}_message_real_id_{sid}.csv"
        cond_msg = np.loadtxt(cond_mp, delimiter=',')
        key = (date, sid)
        cond_lens[key] = cond_book.shape[0]
        gen_books[key] = full_book
        gen_msgs[key]  = np.vstack([cond_msg, gen_msg])
    return gen_books, gen_msgs, cond_lens


def load_all_v2(grid_df):
    all_data = {}
    for _, row in tqdm(grid_df.iterrows(), total=len(grid_df), desc='Loading'):
        try:
            bb, bm, bc = load_folder_data(row['buy_path'],  MAX_SAMPLES, MIDPRICE_MAX)
            sb, sm, sc = load_folder_data(row['sell_path'], MAX_SAMPLES, MIDPRICE_MAX)
            all_data[row['folder']] = {
                'buy':  {'books': bb, 'msgs': bm, 'cond_lens': bc},
                'sell': {'books': sb, 'msgs': sm, 'cond_lens': sc},
            }
        except Exception as e:
            print(f"  ERR {row['folder']}: {e}")
    return all_data

In [5]:
# ── Beta (square-root law) — with insertion_idx ─────────────────

def extract_point_cloud(data, grid_df):
    eps = 1e-12
    rows = []
    for _, grow in grid_df.iterrows():
        folder = grow['folder']
        if folder not in data:
            continue
        d = data[folder]
        mb_val = grow['mb']
        aggr_buy  = load_aggressive_indices(grow['buy_path'])
        aggr_sell = load_aggressive_indices(grow['sell_path'])
        for direction, side, aggr_gen in [('BUY', d['buy'], aggr_buy),
                                           ('SELL', d['sell'], aggr_sell)]:
            if len(aggr_gen) == 0:
                continue
            books, msgs, conds = side['books'], side['msgs'], side['cond_lens']
            for sid in books:
                msg_arr, book_arr = msgs[sid], books[sid]
                junction = conds[sid]
                sample_id = sid[1]
                day = SAMPLE_DAY_MAP[SAMPLE_DAY_MAP['sample_id'] == sample_id]
                if day.empty:
                    continue
                H = float(day.iloc[0]['highest_price']) / TICK_SIZE
                L = float(day.iloc[0]['lowest_price'])  / TICK_SIZE
                V_day = float(day.iloc[0]['execution_sum'])
                if H <= L or L <= 0 or V_day <= eps:
                    continue
                sigma = np.log(H / L) / 0.8325546
                alpha = np.log(max(sigma, eps))
                aggr_idx = junction + aggr_gen
                aggr_idx = aggr_idx[aggr_idx < len(msg_arr)]
                if len(aggr_idx) < 2:
                    continue
                sizes  = msg_arr[aggr_idx, 3].astype(float)
                prices = msg_arr[aggr_idx, 4].astype(float)
                ref = (book_arr[aggr_idx[0], 0] + book_arr[aggr_idx[0], 2]) / 2
                if ref <= 0:
                    continue
                Q_cum = np.cumsum(sizes)
                vwap  = np.cumsum(sizes * prices) / np.maximum(Q_cum, eps)
                imp   = np.abs((vwap - ref) / ref) if direction == 'BUY' else np.abs((ref - vwap) / ref)
                for a in range(len(aggr_idx)):
                    if imp[a] > eps:
                        rows.append({'x': np.log(Q_cum[a] / V_day),
                                     'y': np.log(imp[a]),
                                     'alpha': alpha,
                                     'sample_id': sample_id,
                                     'folder': folder, 'direction': direction,
                                     'mb': mb_val,
                                     'insertion_idx': a})
    if not rows:
        return pd.DataFrame(columns=['x', 'y', 'alpha', 'sample_id', 'folder',
                                     'direction', 'mb', 'insertion_idx'])
    return pd.DataFrame(rows)


def compute_global_beta(df):
    if df.empty:
        return {'beta': np.nan, 'r2': np.nan, 'n': 0}
    y_adj = df['y'].values - df['alpha'].values
    x = df['x'].values
    ok = np.isfinite(x) & np.isfinite(y_adj) & (x != 0)
    xv, yv = x[ok], y_adj[ok]
    if len(xv) < 2:
        return {'beta': np.nan, 'r2': np.nan, 'n': 0}
    beta = float(np.dot(xv, yv) / np.dot(xv, xv))
    ss_res = np.sum((yv - beta * xv) ** 2)
    ss_tot = np.sum(yv ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return {'beta': beta, 'r2': r2, 'n': int(ok.sum())}


def bootstrap_beta(df, n_boot=1000):
    if df.empty:
        return np.array([])
    pc = df[['x', 'y', 'alpha', 'sample_id']].copy()
    pc['y_adj'] = pc['y'] - pc['alpha']
    ok = np.isfinite(pc['x']) & np.isfinite(pc['y_adj']) & (pc['x'] != 0)
    pc = pc[ok]
    groups = {sid: g[['x', 'y_adj']].values for sid, g in pc.groupby('sample_id')}
    ids = np.array(list(groups.keys()))
    n = len(ids)
    if n == 0:
        return np.array([])
    rng = np.random.RandomState(42)
    betas = np.zeros(n_boot)
    for b in range(n_boot):
        chosen = rng.choice(ids, size=n, replace=True)
        pool = np.vstack([groups[s] for s in chosen])
        x, y = pool[:, 0], pool[:, 1]
        betas[b] = np.dot(x, y) / np.dot(x, x)
    return betas


# ── Stratified beta (from NB 121) ───────────────────────────────

def compute_beta_for_subset(df):
    y_adj = df['y'].values - df['alpha'].values
    x = df['x'].values
    valid = np.isfinite(x) & np.isfinite(y_adj) & (x != 0)
    x_v, y_v = x[valid], y_adj[valid]
    n = int(np.sum(valid))
    if n < 2:
        return {'beta': np.nan, 'se': np.nan, 'r2': np.nan, 'n': n}
    beta = float(np.dot(x_v, y_v) / np.dot(x_v, x_v))
    y_pred = beta * x_v
    ss_res = np.sum((y_v - y_pred)**2)
    ss_tot = np.sum(y_v**2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    se = np.sqrt(ss_res / max(n-1, 1)) / np.sqrt(np.dot(x_v, x_v))
    return {'beta': beta, 'se': se, 'r2': r2, 'n': n}


def compute_stratified_beta(df, strat_col='mb', exclude_strata=None):
    """Compute beta separately per mb stratum, then average with equal weights."""
    exclude_strata = exclude_strata or set()
    strata = sorted(s for s in df[strat_col].unique() if s not in exclude_strata)
    per_stratum = []
    for s in strata:
        sub = df[df[strat_col] == s]
        result = compute_beta_for_subset(sub)
        if not np.isnan(result['beta']):
            per_stratum.append({'stratum': s, **result})
    if not per_stratum:
        return {'beta_stratified': np.nan, 'se_stratified': np.nan,
                'per_stratum': [], 'n_strata': 0}
    betas = [r['beta'] for r in per_stratum]
    beta_stratified = np.mean(betas)
    se_stratified = np.std(betas, ddof=1) / np.sqrt(len(betas)) if len(betas) > 1 else np.nan
    return {'beta_stratified': beta_stratified, 'se_stratified': se_stratified,
            'per_stratum': per_stratum, 'n_strata': len(per_stratum)}

In [6]:
# ── Master curves, relaxation, gamma ──────────────────────────────

def compute_master_curve(buy_data, sell_data, folder, aggr_gen,
                         u_max=11.0, n_pts=500):
    i, c, mb, V = parse_folder_params_v2(folder)
    if len(aggr_gen) < 2:
        return None
    s_gen = int(aggr_gen[0])
    e_gen = int(aggr_gen[-1])
    L = e_gen - s_gen
    if L == 0:
        return None
    bb, sb = buy_data['books'], sell_data['books']
    if not bb or not sb:
        return None
    min_len = min(min(b.shape[0] for b in bb.values()),
                  min(b.shape[0] for b in sb.values()))
    junction = list(buy_data['cond_lens'].values())[0]
    u_cap = min(u_max, (min_len - 1 - junction - s_gen) / L)
    if u_cap <= 0:
        return None
    u_grid = np.linspace(0, u_cap, n_pts)

    def side_impacts(books, conds):
        imps = []
        for sid, bk in books.items():
            sample_id = sid[1]
            day = SAMPLE_DAY_MAP[SAMPLE_DAY_MAP['sample_id'] == sample_id]
            if day.empty:
                continue
            H = float(day.iloc[0]['highest_price']) / TICK_SIZE
            Lp = float(day.iloc[0]['lowest_price'])  / TICK_SIZE
            if H <= Lp or Lp <= 0:
                continue
            sigma = np.log(H / Lp) / 0.8325546
            if sigma <= 0:
                continue
            j = conds[sid]
            s_abs = j + s_gen
            if s_abs < 1 or s_abs >= min_len:
                continue
            mid = compute_midprice(bk[:min_len])
            ref = mid[s_abs - 1]
            if ref <= 0:
                continue
            raw = (mid[s_abs:min_len] - ref) / (ref * sigma)
            u_raw = np.arange(len(raw)) / L
            imps.append(np.interp(u_grid, u_raw, raw))
        return np.array(imps) if imps else None

    bi = side_impacts(bb, buy_data['cond_lens'])
    si = side_impacts(sb, sell_data['cond_lens'])
    if bi is None or si is None:
        return None
    mean = (np.mean(bi, axis=0) - np.mean(si, axis=0)) / 2
    std  = np.sqrt(np.std(bi, axis=0)**2 + np.std(si, axis=0)**2) / 2
    return {'u_grid': u_grid, 'combined_mean': mean, 'combined_std': std,
            'L': L, 'i': i, 'c': c, 'mb': mb, 'V': V, 'Q': i * V}


def compute_raw_curve(buy_data, sell_data, folder, aggr_gen,
                      u_max=11.0, n_pts=500):
    i, c, mb, V = parse_folder_params_v2(folder)
    if len(aggr_gen) < 2:
        return None
    s_gen = int(aggr_gen[0])
    e_gen = int(aggr_gen[-1])
    L = e_gen - s_gen
    if L == 0:
        return None
    bb, sb = buy_data['books'], sell_data['books']
    if not bb or not sb:
        return None
    min_len = min(min(b.shape[0] for b in bb.values()),
                  min(b.shape[0] for b in sb.values()))
    junction = list(buy_data['cond_lens'].values())[0]
    u_cap = min(u_max, (min_len - 1 - junction - s_gen) / L)
    if u_cap <= 0:
        return None
    u_grid = np.linspace(0, u_cap, n_pts)

    def side_impacts(books, conds):
        imps = []
        for sid, bk in books.items():
            j = conds[sid]
            s_abs = j + s_gen
            if s_abs < 1 or s_abs >= min_len:
                continue
            mid = compute_midprice(bk[:min_len])
            ref = mid[s_abs - 1]
            raw = mid[s_abs:min_len] - ref
            u_raw = np.arange(len(raw)) / L
            imps.append(np.interp(u_grid, u_raw, raw))
        return np.array(imps) if imps else None

    bi = side_impacts(bb, buy_data['cond_lens'])
    si = side_impacts(sb, sell_data['cond_lens'])
    if bi is None or si is None:
        return None
    mean = (np.mean(bi, axis=0) - np.mean(si, axis=0)) / 2
    std  = np.sqrt(np.std(bi, axis=0)**2 + np.std(si, axis=0)**2) / 2
    return {'u_grid': u_grid, 'combined_mean': mean, 'combined_std': std,
            'L': L, 'i': i, 'c': c, 'mb': mb, 'V': V, 'Q': i * V}


def compute_combined_impact(buy_data, sell_data, folder):
    bb, sb = buy_data['books'], sell_data['books']
    if not bb or not sb:
        return None
    min_len = min(min(b.shape[0] for b in bb.values()),
                  min(b.shape[0] for b in sb.values()))
    buy_returns = compute_midprice_returns(bb, min_len)
    sell_returns = compute_midprice_returns(sb, min_len)
    combined = (buy_returns.mean(axis=0) - sell_returns.mean(axis=0)) / 2
    junction = list(buy_data['cond_lens'].values())[0]
    post = combined[junction:]
    if len(post) == 0:
        return None
    pk_idx = junction + np.argmax(post)
    return {'mean': combined, 'junction': junction,
            'peak': float(combined[pk_idx]), 'final': float(combined[-1]),
            'peak_idx': pk_idx,
            'n_buy': buy_returns.shape[0], 'n_sell': sell_returns.shape[0],
            'min_len': min_len}

In [7]:
# ── Stability (3-method vote) ─────────────────────────────────────

TAIL_FRAC   = 0.20
WINDOW_FRAC = 0.15
SLOPE_THRESH = 0.05
TWIN_THRESH  = 0.03
CONV_THRESH  = 95.0


def stability_for_folder(stats, row):
    if stats is None:
        return {'folder': row['folder'], 'stabilized': False, 'votes': 0}
    mean_curve = stats['mean']
    junction   = stats['junction']
    post = mean_curve[junction:]
    pk_local = np.argmax(post)
    peak_val = post[pk_local]
    post_peak = post[pk_local:]
    final_val = post_peak[-1]
    n = len(post_peak)
    if n < 3 or abs(peak_val) < 1e-12:
        return {'folder': row['folder'], 'stabilized': False, 'votes': 0}
    tl = max(int(n * TAIL_FRAC), 3)
    tail = post_peak[-tl:]
    sl = linregress(np.arange(tl, dtype=float), tail)
    slope_ok = abs(sl.slope * tl / peak_val) < SLOPE_THRESH
    w = max(int(n * WINDOW_FRAC), 3)
    if 2 * w <= n:
        twin_ok = abs((np.mean(post_peak[-w:]) - np.mean(post_peak[-2*w:-w])) / peak_val) < TWIN_THRESH
    else:
        twin_ok = False
    exp_ok = False
    try:
        t = np.arange(n, dtype=float)
        def ef(t, A, tau, C):
            return A * np.exp(-t / tau) + C
        popt, _ = curve_fit(ef, t, post_peak,
                            p0=[float(peak_val - final_val), n / 3.0, float(final_val)],
                            maxfev=10000)
        conv = (1.0 - np.exp(-n / popt[1])) * 100 if popt[1] > 0 else 100.0
        exp_ok = conv > CONV_THRESH
    except Exception:
        pass
    votes = slope_ok + twin_ok + exp_ok
    return {'folder': row['folder'], 'stabilized': votes >= 2, 'votes': int(votes)}

In [8]:
# -- STABILITY FIX: peak at u=1.0 (injection endpoint) on raw curves --

def stability_from_raw_curve(u_grid, combined_mean):
    peak_idx = np.searchsorted(u_grid, 1.0)
    if peak_idx >= len(combined_mean) - 2:
        return False, 0
    peak_val = combined_mean[peak_idx]
    post_peak = combined_mean[peak_idx:]
    n = len(post_peak)
    if n < 5 or abs(peak_val) < 1e-12:
        return False, 0
    final_val = post_peak[-1]
    tl = max(int(n * 0.20), 3)
    tail = post_peak[-tl:]
    sl = linregress(np.arange(tl, dtype=float), tail)
    slope_ok = abs(sl.slope * tl / peak_val) < 0.05
    w = max(int(n * 0.15), 3)
    twin_ok = False
    if 2 * w <= n:
        twin_ok = abs((np.mean(post_peak[-w:]) - np.mean(post_peak[-2*w:-w])) / peak_val) < 0.03
    exp_ok = False
    try:
        def exp_model(t, A, tau, C):
            return A * np.exp(-t / tau) + C
        popt, _ = curve_fit(exp_model, np.arange(n, dtype=float), post_peak,
                           p0=[float(peak_val - final_val), n/3.0, float(final_val)],
                           maxfev=10000)
        conv = (1.0 - np.exp(-n / popt[1])) * 100 if popt[1] > 0 else 100.0
        exp_ok = conv > 95.0
    except Exception:
        pass
    votes = slope_ok + twin_ok + exp_ok
    return votes >= 2, int(votes)

print("Stability fix loaded: peak at u=1.0 on raw curves.")

Stability fix loaded: peak at u=1.0 on raw curves.


In [9]:
# ── Decay function fitting ──────────────────────────────────────

def fit_decay(u_grid, mean_curve, u_peak=1.0):
    mask_post = u_grid > u_peak
    if mask_post.sum() < 5:
        return None
    u_post = u_grid[mask_post]
    I_post = mean_curve[mask_post]
    I_final = I_post[-1]
    I_temp = I_post - I_final

    result = {'u_post': u_post, 'I_post': I_post, 'I_final': I_final}

    du = u_post - u_peak
    ok_pl = (du > 0.01) & (I_temp > 1e-12)
    if ok_pl.sum() >= 3:
        try:
            ln_du = np.log(du[ok_pl])
            ln_It = np.log(I_temp[ok_pl])
            sl = linregress(ln_du, ln_It)
            gamma = -sl.slope
            A_pl = np.exp(sl.intercept)
            I_fit_pl = A_pl * du**(-gamma) + I_final
            ss_res_pl = np.sum((I_post[ok_pl] - (A_pl * du[ok_pl]**(-gamma) + I_final))**2)
            ss_tot_pl = np.sum((I_post[ok_pl] - np.mean(I_post[ok_pl]))**2)
            r2_pl = 1 - ss_res_pl / ss_tot_pl if ss_tot_pl > 0 else 0
            k_pl = 2
            n_pl = int(ok_pl.sum())
            aic_pl = n_pl * np.log(ss_res_pl / n_pl + 1e-30) + 2 * k_pl
            result['gamma'] = gamma
            result['A_pl'] = A_pl
            result['r2_pl'] = r2_pl
            result['aic_pl'] = aic_pl
            result['I_fit_pl'] = I_fit_pl
        except Exception:
            pass

    try:
        def exp_decay(u, A, tau, C):
            return A * np.exp(-(u - u_peak) / tau) + C
        p0 = [float(I_post[0] - I_final), 0.5, float(I_final)]
        popt, _ = curve_fit(exp_decay, u_post, I_post, p0=p0, maxfev=10000)
        I_fit_exp = exp_decay(u_post, *popt)
        ss_res_exp = np.sum((I_post - I_fit_exp)**2)
        ss_tot_exp = np.sum((I_post - np.mean(I_post))**2)
        r2_exp = 1 - ss_res_exp / ss_tot_exp if ss_tot_exp > 0 else 0
        k_exp = 3
        n_exp = len(u_post)
        aic_exp = n_exp * np.log(ss_res_exp / n_exp + 1e-30) + 2 * k_exp
        result['tau'] = popt[1]
        result['r2_exp'] = r2_exp
        result['aic_exp'] = aic_exp
        result['I_fit_exp'] = I_fit_exp
    except Exception:
        pass

    return result

In [10]:
# -- Cell 9: Load & process all 4 checkpoints --

R = OrderedDict()

for label, cfg in CHECKPOINTS.items():
    step = cfg['step']
    buy_p  = EVAL_BASE / cfg['key'] / 'context_500_buy'
    sell_p = EVAL_BASE / cfg['key'] / 'context_500_sell'
    if not buy_p.exists() or not sell_p.exists():
        print(f"SKIP {label}: paths not found")
        continue
    grid = discover_v2_folders(buy_p, sell_p)
    if grid.empty:
        print(f"SKIP {label}: no folders discovered")
        continue
    print(f"\n{'='*60}\n  {label} (step={step}): {len(grid)} configs")
    data = load_all_v2(grid)
    print(f"  Loaded {len(data)} folders")

    # -- Beta (exclude mb=20) --
    pc_all = extract_point_cloud(data, grid)
    pc = pc_all[pc_all['mb'] != 20] if len(pc_all) > 0 else pc_all
    print(f"  Point cloud: {len(pc_all)} total, {len(pc)} after mb!=20 filter")
    bstat = compute_global_beta(pc)
    bbetas = bootstrap_beta(pc, N_BOOTSTRAP)

    # -- Stratified beta --
    strat = compute_stratified_beta(pc, strat_col='mb', exclude_strata={20})

    # -- Sigma-normalized master curves --
    curves = {}
    for _, row in grid.iterrows():
        f = row['folder']
        if f not in data:
            continue
        aggr = load_aggressive_indices(row['buy_path'])
        cv = compute_master_curve(data[f]['buy'], data[f]['sell'], f, aggr)
        if cv is not None:
            curves[f] = cv

    # -- Raw curves --
    raw_curves = {}
    for _, row in grid.iterrows():
        f = row['folder']
        if f not in data:
            continue
        aggr = load_aggressive_indices(row['buy_path'])
        cv = compute_raw_curve(data[f]['buy'], data[f]['sell'], f, aggr)
        if cv is not None:
            raw_curves[f] = cv

    # -- Combined impact -> stability (ORIGINAL: argmax peak) --
    stab_rows, metrics_rows = [], []
    for _, row in grid.iterrows():
        f = row['folder']
        if f not in data:
            continue
        st = compute_combined_impact(data[f]['buy'], data[f]['sell'], f)
        stab_rows.append(stability_for_folder(st, row))
        if st is not None:
            metrics_rows.append({'folder': f, 'i': row['i'], 'mb': row['mb'],
                                 'V': row['V'], 'peak': st['peak']})
    stab_df = pd.DataFrame(stab_rows)
    metrics_df = pd.DataFrame(metrics_rows) if metrics_rows else pd.DataFrame()

    # -- Gamma (V-scaling) --
    gamma_rows = []
    if not metrics_df.empty:
        for (iv, mbv), grp in metrics_df.groupby(['i', 'mb']):
            grp = grp.sort_values('V')
            if len(grp) < 3:
                continue
            Vs = grp['V'].values.astype(float)
            pks = grp['peak'].values
            if np.any(pks <= 0) or np.any(Vs <= 0):
                continue
            sl = linregress(np.log(Vs), np.log(pks))
            gamma_rows.append({'i': iv, 'mb': mbv, 'gamma': sl.slope,
                               'r2': sl.rvalue**2, 'se': sl.stderr})
    gamma_df = pd.DataFrame(gamma_rows) if gamma_rows else pd.DataFrame()

    # -- Relaxation ratios (from RAW curves) --
    relax_rows = []
    for f, cv in raw_curves.items():
        u, m = cv['u_grid'], cv['combined_mean']
        I_peak = float(np.interp(1.0, u, m))
        if abs(I_peak) < 1e-12:
            continue
        I_final = float(m[-1])
        relax_rows.append({'folder': f, 'I_peak': I_peak, 'I_final': I_final,
                           'ratio': I_final / I_peak,
                           'mb': cv['mb'], 'V': cv['V'],
                           'i': cv['i'], 'Q': cv['Q']})
    relax_df = pd.DataFrame(relax_rows) if relax_rows else pd.DataFrame()

    # -- FIXED relaxation: at u=3.0 + end --
    relax_fixed_rows = []
    for f, cv in raw_curves.items():
        u, m = cv['u_grid'], cv['combined_mean']
        I_peak = float(np.interp(1.0, u, m))
        if abs(I_peak) < 1e-12:
            continue
        I_at_u3 = float(np.interp(3.0, u, m))
        I_final_end = float(m[-1])
        relax_fixed_rows.append({
            'folder': f,
            'I_peak': I_peak,
            'I_final_u3': I_at_u3,
            'I_final_end': I_final_end,
            'ratio_u3': I_at_u3 / I_peak,
            'ratio_end': I_final_end / I_peak,
            'mb': cv['mb'], 'V': cv['V'], 'i': cv['i'], 'Q': cv['Q'],
        })
    relax_df_fixed = pd.DataFrame(relax_fixed_rows) if relax_fixed_rows else pd.DataFrame()

    # -- FIXED stability: peak at u=1.0 on raw curves --
    stab_fixed_rows = []
    for f, cv in raw_curves.items():
        stable, votes = stability_from_raw_curve(cv['u_grid'], cv['combined_mean'])
        stab_fixed_rows.append({'folder': f, 'stabilized': stable, 'votes': votes})
    stab_df_fixed = pd.DataFrame(stab_fixed_rows)

    R[label] = {
        'grid': grid, 'pc': pc, 'beta': bstat, 'boot': bbetas,
        'strat': strat,
        'step': step,
        'curves': curves, 'raw_curves': raw_curves,
        'relax_df': relax_df,
        'relax_df_fixed': relax_df_fixed,
        'stab_df': stab_df,
        'stab_df_fixed': stab_df_fixed,
        'gamma_df': gamma_df,
    }
    stable_frac = stab_df['stabilized'].mean() if not stab_df.empty else 0
    stable_frac_fix = stab_df_fixed['stabilized'].mean() if not stab_df_fixed.empty else 0
    relax_med = relax_df['ratio'].median() if not relax_df.empty else np.nan
    relax_med_u3 = relax_df_fixed['ratio_u3'].median() if not relax_df_fixed.empty else np.nan
    print(f"  beta={bstat['beta']:.4f}  R2={bstat['r2']:.4f}  n={bstat['n']:,}")
    print(f"  beta_strat={strat['beta_stratified']:.4f}  (n_strata={strat['n_strata']})")
    print(f"  Curves: {len(curves)} sigma-norm, {len(raw_curves)} raw")
    print(f"  Relax median: end={relax_med:.3f}, u3={relax_med_u3:.3f}")
    print(f"  Stable: orig={stable_frac:.0%}, fix(u=1)={stable_frac_fix:.0%}")
    print(f"  Gamma: {len(gamma_rows)} configs")

    del data
    gc.collect()

print(f"\n{'='*60}\nLoaded {len(R)} checkpoints: {list(R.keys())}")


  Step 22820 (step=22820): 30 configs


Loading: 100%|██████████| 30/30 [00:04<00:00,  6.74it/s]


  Loaded 30 folders
  Point cloud: 3755 total, 3534 after mb!=20 filter
  beta=0.5259  R2=0.9268  n=3,534
  beta_strat=0.5132  (n_strata=3)
  Curves: 27 sigma-norm, 27 raw
  Relax median: end=0.655, u3=0.868
  Stable: orig=0%, fix(u=1)=0%
  Gamma: 10 configs

  Step 38335 (step=38335): 30 configs


Loading: 100%|██████████| 30/30 [00:03<00:00,  8.01it/s]


  Loaded 30 folders
  Point cloud: 3746 total, 3525 after mb!=20 filter
  beta=0.5263  R2=0.9316  n=3,525
  beta_strat=0.5190  (n_strata=3)
  Curves: 27 sigma-norm, 27 raw
  Relax median: end=0.826, u3=0.928
  Stable: orig=0%, fix(u=1)=0%
  Gamma: 10 configs

  Step 79688 (step=79688): 30 configs


Loading: 100%|██████████| 30/30 [00:03<00:00,  7.58it/s]


  Loaded 30 folders
  Point cloud: 3812 total, 3587 after mb!=20 filter
  beta=0.5552  R2=0.9464  n=3,587
  beta_strat=0.5465  (n_strata=3)
  Curves: 27 sigma-norm, 27 raw
  Relax median: end=0.704, u3=0.772
  Stable: orig=0%, fix(u=1)=0%
  Gamma: 10 configs

  Step 100378 (step=100378): 30 configs


Loading: 100%|██████████| 30/30 [00:03<00:00,  8.49it/s]


  Loaded 30 folders
  Point cloud: 3751 total, 3524 after mb!=20 filter
  beta=0.5458  R2=0.9389  n=3,524
  beta_strat=0.5415  (n_strata=3)
  Curves: 27 sigma-norm, 27 raw
  Relax median: end=0.877, u3=0.842
  Stable: orig=0%, fix(u=1)=0%
  Gamma: 10 configs

Loaded 4 checkpoints: ['Step 22820', 'Step 38335', 'Step 79688', 'Step 100378']


---
## Part 2: Beta Evolution

Does the square-root law exponent converge to the theoretical value beta = 0.5 with more training?

In [11]:
# -- Table: Beta global + stratified + CI per step --

rows = []
for label, r in R.items():
    b = r['beta']
    ci = np.percentile(r['boot'], [2.5, 97.5]) if len(r['boot']) > 0 else [np.nan, np.nan]
    s = r['strat']
    rows.append({
        'Checkpoint': label,
        'Step': r['step'],
        'beta_global': f"{b['beta']:.3f}",
        'R2': f"{b['r2']:.3f}",
        'N': f"{b['n']:,}",
        '95% CI': f"[{ci[0]:.3f}, {ci[1]:.3f}]",
        'beta_strat': f"{s['beta_stratified']:.3f}",
        'n_strata': s['n_strata'],
    })
table_beta = pd.DataFrame(rows)
print("\n-- Beta: Global + Stratified per Checkpoint --")
print(table_beta.to_string(index=False))


-- Beta: Global + Stratified per Checkpoint --
 Checkpoint   Step beta_global    R2     N         95% CI beta_strat  n_strata
 Step 22820  22820       0.526 0.927 3,534 [0.486, 0.563]      0.513         3
 Step 38335  38335       0.526 0.932 3,525 [0.490, 0.559]      0.519         3
 Step 79688  79688       0.555 0.946 3,587 [0.528, 0.581]      0.546         3
Step 100378 100378       0.546 0.939 3,524 [0.517, 0.575]      0.541         3


In [12]:
# -- KEY FIGURE: Beta vs Training Step (line + CI band + theory 0.5) --

steps_arr = np.array([r['step'] for r in R.values()])
betas_arr = np.array([r['beta']['beta'] for r in R.values()])
betas_strat = np.array([r['strat']['beta_stratified'] for r in R.values()])
ci_lo = np.array([np.percentile(r['boot'], 2.5) if len(r['boot']) > 0 else np.nan for r in R.values()])
ci_hi = np.array([np.percentile(r['boot'], 97.5) if len(r['boot']) > 0 else np.nan for r in R.values()])

fig = go.Figure()

# CI band
fig.add_trace(go.Scatter(
    x=np.concatenate([steps_arr, steps_arr[::-1]]),
    y=np.concatenate([ci_hi, ci_lo[::-1]]),
    fill='toself', fillcolor='rgba(31,120,180,0.15)',
    line=dict(width=0), showlegend=False))

# Global beta line
fig.add_trace(go.Scatter(
    x=steps_arr, y=betas_arr, mode='lines+markers',
    line=dict(color='#1f78b4', width=2.5),
    marker=dict(size=10, color=[STEP_COLORS[l] for l in R.keys()],
                line=dict(color='black', width=1.5)),
    name='Global beta'))

# Stratified beta line
fig.add_trace(go.Scatter(
    x=steps_arr, y=betas_strat, mode='lines+markers',
    line=dict(color='#ff7f0e', width=2, dash='dash'),
    marker=dict(size=8, symbol='diamond',
                color=[STEP_COLORS[l] for l in R.keys()],
                line=dict(color='black', width=1)),
    name='Stratified beta'))

# Theory line
fig.add_hline(y=0.5, line_dash='dash', line_color='black', line_width=1.5,
              annotation_text='Theory (0.5)', annotation_font_size=11,
              annotation_position='bottom right')

pub_layout(fig, width=FULL_W, height=450, legend_pos='tr')
fig.update_xaxes(title_text='Training step')
fig.update_yaxes(title_text='beta')
save_fig(fig, 'beta_vs_step.png')
fig.show()

  Saved: beta_vs_step.png


In [13]:
# -- Figure: Beta Regression Lines (4 checkpoints overlaid) --

fig = go.Figure()
x_range = np.array([-16, -4])

for label, r in R.items():
    pc = r['pc']
    if pc.empty:
        continue
    color = STEP_COLORS[label]
    beta  = r['beta']['beta']
    n_show = min(5000, len(pc))
    idx = np.random.RandomState(42).choice(len(pc), n_show, replace=False)
    sub = pc.iloc[idx]
    fig.add_trace(go.Scatter(
        x=sub['x'], y=sub['y'] - sub['alpha'], mode='markers',
        marker=dict(size=2.5, color=color, opacity=0.12),
        name=label, showlegend=False))
    fig.add_trace(go.Scatter(
        x=x_range, y=beta * x_range, mode='lines',
        line=dict(color=color, width=2.5),
        name=f"{label} (beta={beta:.3f})"))

fig.add_trace(go.Scatter(
    x=x_range, y=0.5 * x_range, mode='lines',
    line=dict(color='black', width=1.5, dash='dash'),
    name='Theory (beta=0.5)'))

pub_layout(fig, width=FULL_W, height=480, legend_pos='br')
fig.update_xaxes(title_text='ln(Q / V)')
fig.update_yaxes(title_text='ln(I / sigma)')
save_fig(fig, 'beta_regression_4ckpt.png')
fig.show()

  Saved: beta_regression_4ckpt.png


In [14]:
# -- Figure: Bootstrap Beta Distributions (4 checkpoints overlaid) --

fig = go.Figure()
for label, r in R.items():
    bb = r['boot']
    if len(bb) == 0:
        continue
    fig.add_trace(go.Histogram(
        x=bb, nbinsx=50, name=label, opacity=0.55,
        marker_color=STEP_COLORS[label],
        marker_line_color='black', marker_line_width=0.5))

fig.add_vline(x=0.5, line_dash='dash', line_color='black', line_width=1.5,
              annotation_text='beta = 0.5', annotation_font_size=11,
              annotation_position='top left')

pub_layout(fig, width=FULL_W, height=400, legend_pos='tr',
           barmode='overlay')
fig.update_xaxes(title_text='beta (bootstrap)')
fig.update_yaxes(title_text='Count')
save_fig(fig, 'bootstrap_beta_4ckpt.png')
fig.show()

  Saved: bootstrap_beta_4ckpt.png


In [15]:
# -- KEY FIGURE: Per-Stratum Beta vs Training Step --

fig = go.Figure()

# Collect all mb strata across checkpoints
all_strata = set()
for r in R.values():
    for ps in r['strat']['per_stratum']:
        all_strata.add(ps['stratum'])
all_strata = sorted(all_strata)

mb_palette = px.colors.qualitative.Set2
for si, mb_val in enumerate(all_strata):
    x_vals, y_vals = [], []
    for label, r in R.items():
        for ps in r['strat']['per_stratum']:
            if ps['stratum'] == mb_val:
                x_vals.append(r['step'])
                y_vals.append(ps['beta'])
    if x_vals:
        fig.add_trace(go.Scatter(
            x=x_vals, y=y_vals, mode='lines+markers',
            line=dict(color=mb_palette[si % len(mb_palette)], width=2),
            marker=dict(size=8),
            name=f'mb={mb_val}'))

fig.add_hline(y=0.5, line_dash='dash', line_color='black', line_width=1.5,
              annotation_text='Theory (0.5)', annotation_font_size=11,
              annotation_position='bottom right')

pub_layout(fig, width=FULL_W, height=450, legend_pos='tr')
fig.update_xaxes(title_text='Training step')
fig.update_yaxes(title_text='beta (per stratum)')
save_fig(fig, 'per_stratum_beta_vs_step.png')
fig.show()

  Saved: per_stratum_beta_vs_step.png


---
## Part 3: Decay Evolution

How do master curves, relaxation ratios, and decay exponents evolve across training?

In [16]:
# -- Figure: Master Curves faceted 2x2 (one per checkpoint) --

_u_all = []
for label, r in R.items():
    for f, c in r['curves'].items():
        _u_all.append(c['u_grid'][-1])
U_MASTER = min(3.0, np.percentile(_u_all, 10)) if _u_all else 3.0
print(f"  Master curves plot limit: u <= {U_MASTER:.2f}")

n_ckpt = len(R)
n_cols = 2
n_rows = math.ceil(n_ckpt / n_cols)
fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f'<b>{l}</b>' for l in R.keys()],
    horizontal_spacing=0.10, vertical_spacing=0.10)

palette = px.colors.qualitative.D3 + px.colors.qualitative.Set2

for idx, (label, r) in enumerate(R.items()):
    row, col = idx // n_cols + 1, idx % n_cols + 1
    sorted_f = sorted(r['curves'].keys(),
        key=lambda f: (parse_folder_params_v2(f)[2], parse_folder_params_v2(f)[0]))
    for fi, folder in enumerate(sorted_f):
        c = r['curves'][folder]
        u, m = c['u_grid'], c['combined_mean']
        mask = u <= U_MASTER
        fig.add_trace(go.Scatter(
            x=u[mask], y=m[mask], mode='lines',
            line=dict(color=palette[fi % len(palette)], width=1.2),
            showlegend=False), row=row, col=col)
    fig.add_vline(x=1.0, line_dash='dot', line_color='rgba(0,0,0,0.35)',
                  line_width=1, row=row, col=col)

fig.update_layout(
    width=FULL_W, height=int(FULL_W * 0.36 * n_rows),
    template='plotly_white',
    font=dict(family='Times New Roman, DejaVu Serif, serif', size=12, color='black'),
    title=None,
    margin=dict(l=55, r=15, t=35, b=50),
)
fig.update_xaxes(**_AX, title_text='u = n / L', title_font_size=12, tickfont_size=10)
fig.update_yaxes(**_AX, title_text='I_norm(u)', title_font_size=12, tickfont_size=10)
fig.update_annotations(font_size=13)
save_fig(fig, 'master_curves_facet_4ckpt.png')
fig.show()

  Master curves plot limit: u <= 3.00
  Saved: master_curves_facet_4ckpt.png


In [17]:
# -- KEY FIGURE: Average Master Curve overlay (4 checkpoints) --

fig = go.Figure()

u_max_all = []
for label, r in R.items():
    for f, c in r['curves'].items():
        u_max_all.append(c['u_grid'][-1])
U_PLOT = min(3.0, np.percentile(u_max_all, 10)) if u_max_all else 3.0
u_common = np.linspace(0, U_PLOT, 300)
print(f"  Average master curve u_max = {U_PLOT:.2f}")

for label, r in R.items():
    interps = []
    for f, c in r['curves'].items():
        if c['u_grid'][-1] >= U_PLOT:
            interps.append(np.interp(u_common, c['u_grid'], c['combined_mean']))
    if not interps:
        print(f"  WARNING: {label} -- no curves reach u={U_PLOT:.2f}, skipping")
        continue
    avg = np.mean(interps, axis=0)
    std = np.std(interps, axis=0)
    color = STEP_COLORS[label]
    rc, gc_, bc = int(color[1:3],16), int(color[3:5],16), int(color[5:7],16)
    fill_rgba = f'rgba({rc},{gc_},{bc},0.12)'

    fig.add_trace(go.Scatter(
        x=np.concatenate([u_common, u_common[::-1]]),
        y=np.concatenate([avg + std, (avg - std)[::-1]]),
        fill='toself', fillcolor=fill_rgba,
        line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(
        x=u_common, y=avg, mode='lines',
        line=dict(color=color, width=2.5), name=label))

fig.add_vline(x=1.0, line_dash='dot', line_color='rgba(0,0,0,0.35)', line_width=1)

pub_layout(fig, width=FULL_W, height=420, legend_pos='tr')
fig.update_xaxes(title_text='u = n / L')
fig.update_yaxes(title_text='I_norm(u)')
save_fig(fig, 'avg_master_curve_4ckpt.png')
fig.show()

  Average master curve u_max = 3.00
  Saved: avg_master_curve_4ckpt.png


In [18]:
# -- KEY FIGURE: Relaxation ratio vs Training Step --

steps_relax = []
relax_meds = []
relax_q25 = []
relax_q75 = []

for label, r in R.items():
    rdf = r['relax_df_fixed']
    if rdf.empty:
        continue
    steps_relax.append(r['step'])
    relax_meds.append(rdf['ratio_u3'].median())
    relax_q25.append(rdf['ratio_u3'].quantile(0.25))
    relax_q75.append(rdf['ratio_u3'].quantile(0.75))

steps_relax = np.array(steps_relax)
relax_meds = np.array(relax_meds)
relax_q25 = np.array(relax_q25)
relax_q75 = np.array(relax_q75)

fig = go.Figure()

# IQR band
fig.add_trace(go.Scatter(
    x=np.concatenate([steps_relax, steps_relax[::-1]]),
    y=np.concatenate([relax_q75, relax_q25[::-1]]),
    fill='toself', fillcolor='rgba(31,120,180,0.15)',
    line=dict(width=0), showlegend=False))

# Median line
fig.add_trace(go.Scatter(
    x=steps_relax, y=relax_meds, mode='lines+markers',
    line=dict(color='#1f78b4', width=2.5),
    marker=dict(size=10, color=[STEP_COLORS[l] for l in R.keys() if not R[l]['relax_df_fixed'].empty],
                line=dict(color='black', width=1.5)),
    name='Median r@u=3'))

fig.add_hline(y=2/3, line_dash='dash', line_color='black', line_width=1.5,
              annotation_text='Theory (2/3)', annotation_font_size=11,
              annotation_position='bottom right')

pub_layout(fig, width=FULL_W, height=450, legend_pos='tr')
fig.update_xaxes(title_text='Training step')
fig.update_yaxes(title_text='Relaxation ratio I(u=3)/I(u=1)')
save_fig(fig, 'relaxation_vs_step.png')
fig.show()

  Saved: relaxation_vs_step.png


In [19]:
# -- Figure: Relaxation box plots per checkpoint --

fig = go.Figure()
for label, r in R.items():
    rdf = r['relax_df_fixed']
    if rdf.empty:
        continue
    fig.add_trace(go.Box(
        y=rdf['ratio_u3'], name=label,
        marker_color=STEP_COLORS[label],
        line_color=STEP_COLORS[label],
        boxpoints='all', jitter=0.3, pointpos=-1.5,
        marker=dict(size=4, opacity=0.5),
        line_width=1.5))

fig.add_hline(y=2/3, line_dash='dash', line_color='black', line_width=1.5,
              annotation_text='2/3', annotation_font_size=11,
              annotation_position='bottom right')

pub_layout(fig, width=FULL_W, height=400, legend_pos='none')
fig.update_xaxes(title_text='', tickangle=-30)
fig.update_yaxes(title_text='I_final(u=3) / I_peak(u=1)')
save_fig(fig, 'relaxation_boxplot_4ckpt.png')
fig.show()

  Saved: relaxation_boxplot_4ckpt.png


In [20]:
# -- Decay Function Fitting (per checkpoint) --

decay_results = OrderedDict()

for label, r in R.items():
    fits = []
    for f, cv in r['curves'].items():
        dr = fit_decay(cv['u_grid'], cv['combined_mean'])
        if dr is not None and 'gamma' in dr:
            fits.append({
                'folder': f, 'gamma': dr['gamma'],
                'r2_pl': dr.get('r2_pl', np.nan),
                'r2_exp': dr.get('r2_exp', np.nan),
                'aic_pl': dr.get('aic_pl', np.nan),
                'aic_exp': dr.get('aic_exp', np.nan),
                'tau': dr.get('tau', np.nan),
            })
    decay_df = pd.DataFrame(fits) if fits else pd.DataFrame()
    decay_results[label] = decay_df

print("\n-- Decay Fitting Summary --")
for label, ddf in decay_results.items():
    if ddf.empty:
        continue
    g = ddf['gamma']
    print(f"  {label:15s}  gamma={g.mean():.3f}+/-{g.std():.3f} (med={g.median():.3f})  "
          f"AIC_PL<Exp: {(ddf['aic_pl'] < ddf['aic_exp']).sum()}/{len(ddf)}")


-- Decay Fitting Summary --
  Step 22820       gamma=0.332+/-0.629 (med=0.482)  AIC_PL<Exp: 0/27
  Step 38335       gamma=0.031+/-1.481 (med=0.393)  AIC_PL<Exp: 0/25
  Step 79688       gamma=-3.782+/-15.396 (med=0.283)  AIC_PL<Exp: 0/27
  Step 100378      gamma=-0.288+/-1.902 (med=0.344)  AIC_PL<Exp: 0/27


In [21]:
# -- KEY FIGURE: Decay exponent (gamma) vs Training Step --

steps_g = []
gamma_meds = []
gamma_q25 = []
gamma_q75 = []
gamma_colors = []

for label, r in R.items():
    ddf = decay_results[label]
    if ddf.empty:
        continue
    steps_g.append(r['step'])
    gamma_meds.append(ddf['gamma'].median())
    gamma_q25.append(ddf['gamma'].quantile(0.25))
    gamma_q75.append(ddf['gamma'].quantile(0.75))
    gamma_colors.append(STEP_COLORS[label])

steps_g = np.array(steps_g)
gamma_meds = np.array(gamma_meds)
gamma_q25 = np.array(gamma_q25)
gamma_q75 = np.array(gamma_q75)

fig = go.Figure()

# IQR band
fig.add_trace(go.Scatter(
    x=np.concatenate([steps_g, steps_g[::-1]]),
    y=np.concatenate([gamma_q75, gamma_q25[::-1]]),
    fill='toself', fillcolor='rgba(31,120,180,0.15)',
    line=dict(width=0), showlegend=False))

# Median line
fig.add_trace(go.Scatter(
    x=steps_g, y=gamma_meds, mode='lines+markers',
    line=dict(color='#1f78b4', width=2.5),
    marker=dict(size=10, color=gamma_colors,
                line=dict(color='black', width=1.5)),
    name='Median gamma'))

# Reference band [0.5, 0.8]
fig.add_hrect(y0=0.5, y1=0.8, fillcolor='rgba(0,200,0,0.08)',
              line=dict(width=0))
fig.add_hline(y=0.5, line_dash='dot', line_color='grey', line_width=1)
fig.add_hline(y=0.8, line_dash='dot', line_color='grey', line_width=1,
              annotation_text='[0.5, 0.8]', annotation_font_size=10,
              annotation_position='top right')

pub_layout(fig, width=FULL_W, height=450, legend_pos='tr')
fig.update_xaxes(title_text='Training step')
fig.update_yaxes(title_text='gamma (decay exponent)')
save_fig(fig, 'gamma_vs_step.png')
fig.show()

  Saved: gamma_vs_step.png


In [22]:
# -- Figure: Decay exponent box plots per checkpoint --

fig = go.Figure()
for label, ddf in decay_results.items():
    if ddf.empty:
        continue
    fig.add_trace(go.Box(
        y=ddf['gamma'], name=label,
        marker_color=STEP_COLORS[label],
        line_color=STEP_COLORS[label],
        boxpoints='all', jitter=0.3, pointpos=-1.5,
        marker=dict(size=4, opacity=0.5),
        line_width=1.5))

fig.add_hline(y=0.5, line_dash='dash', line_color='black', line_width=1,
              annotation_text='gamma=0.5', annotation_font_size=10,
              annotation_position='bottom right')
fig.add_hline(y=0.8, line_dash='dash', line_color='grey', line_width=1,
              annotation_text='gamma=0.8', annotation_font_size=10,
              annotation_position='top right')

pub_layout(fig, width=FULL_W, height=400, legend_pos='none')
fig.update_xaxes(title_text='', tickangle=-30)
fig.update_yaxes(title_text='gamma (decay exponent)')
save_fig(fig, 'decay_boxplot_4ckpt.png')
fig.show()

  Saved: decay_boxplot_4ckpt.png


---
## Part 4: Stability

Does the fraction of stable folders increase with training?

In [23]:
# -- KEY FIGURE: Stability fraction vs Training Step (bar) --

labels_s = list(R.keys())
fracs_s = []
for label in labels_s:
    sdf = R[label]['stab_df']
    fracs_s.append(sdf['stabilized'].mean() if not sdf.empty else 0)
colors_s = [STEP_COLORS[l] for l in labels_s]

fig = go.Figure(go.Bar(
    x=labels_s, y=fracs_s, marker_color=colors_s, width=0.55,
    marker_line_color='black', marker_line_width=1))

fig.add_hline(y=0.5, line_dash='dot', line_color='rgba(0,0,0,0.35)', line_width=1)

pub_layout(fig, width=FULL_W, height=400, legend_pos='none',
           yaxis_range=[0, 1.05])
fig.update_xaxes(title_text='', tickangle=-30)
fig.update_yaxes(title_text='Fraction stable (>=2/3 votes)')
save_fig(fig, 'stability_vs_step.png')
fig.show()

# Also print table
print("\n-- Stability per Checkpoint --")
for label, frac in zip(labels_s, fracs_s):
    sdf = R[label]['stab_df']
    n_stable = int(sdf['stabilized'].sum()) if not sdf.empty else 0
    n_total = len(sdf)
    print(f"  {label:15s}  {n_stable}/{n_total} = {frac:.0%}")

  Saved: stability_vs_step.png



-- Stability per Checkpoint --
  Step 22820       0/30 = 0%
  Step 38335       0/30 = 0%
  Step 79688       0/30 = 0%
  Step 100378      0/30 = 0%


---
## Part 5: Joint Evolution

All metrics combined: how does the overall quality evolve?

In [24]:
# -- Grand Summary Table (all metrics per checkpoint) --

print("=" * 100)
hdr = (f"{'Checkpoint':15s}  {'Step':>7s}  {'beta':>6s}  {'b_strat':>7s}  {'R2':>6s}  "
       f"{'Relax':>6s}  {'Rlx@u3':>7s}  {'Stable':>7s}  "
       f"{'gamma_d':>7s}  {'gamma_V':>7s}")
print(hdr)
print("-" * 100)
for label, r in R.items():
    beta = r['beta']['beta']
    beta_s = r['strat']['beta_stratified']
    r2 = r['beta']['r2']
    relax_med = r['relax_df']['ratio'].median() if not r['relax_df'].empty else np.nan
    relax_u3 = r['relax_df_fixed']['ratio_u3'].median() if not r['relax_df_fixed'].empty else np.nan
    stable = r['stab_df']['stabilized'].mean() if not r['stab_df'].empty else 0

    ddf = decay_results.get(label, pd.DataFrame())
    gamma_dec = ddf['gamma'].median() if not ddf.empty else np.nan
    gamma_v = r['gamma_df']['gamma'].mean() if not r['gamma_df'].empty else np.nan

    print(f"{label:15s}  {r['step']:7d}  {beta:6.3f}  {beta_s:7.3f}  {r2:6.3f}  "
          f"{relax_med:6.3f}  {relax_u3:7.3f}  {stable:6.0%}  "
          f"{gamma_dec:7.3f}  {gamma_v:7.3f}")
print("=" * 100)
print(f"{'Theory':15s}  {'':>7s}  {'0.500':>6s}  {'0.500':>7s}  {'':>6s}  "
      f"{'0.667':>6s}  {'0.667':>7s}  {'':>7s}  "
      f"{'0.5-8':>7s}  {'':>7s}")

Checkpoint          Step    beta  b_strat      R2   Relax   Rlx@u3   Stable  gamma_d  gamma_V
----------------------------------------------------------------------------------------------------
Step 22820         22820   0.526    0.513   0.927   0.655    0.868      0%    0.482    0.764
Step 38335         38335   0.526    0.519   0.932   0.826    0.928      0%    0.393    0.678
Step 79688         79688   0.555    0.546   0.946   0.704    0.772      0%    0.283    0.611
Step 100378       100378   0.546    0.541   0.939   0.877    0.842      0%    0.344    0.571
Theory                     0.500    0.500           0.667    0.667             0.5-8         


In [25]:
# -- KEY FIGURE: Beta vs Decay Ratio trajectory (2D scatter + arrows) --

fig = go.Figure()

traj_steps = []
traj_betas = []
traj_relax = []
traj_colors = []

for label, r in R.items():
    rdf = r['relax_df_fixed']
    if rdf.empty:
        continue
    traj_steps.append(r['step'])
    traj_betas.append(r['beta']['beta'])
    traj_relax.append(rdf['ratio_u3'].median())
    traj_colors.append(STEP_COLORS[label])

# Acceptable zone
fig.add_shape(type='rect', x0=0.3, x1=0.7, y0=0.5, y1=1.0,
              fillcolor='rgba(0,200,0,0.08)', line=dict(color='green', width=1, dash='dot'))

# Theory point
fig.add_trace(go.Scatter(
    x=[0.5], y=[2/3], mode='markers',
    marker=dict(symbol='star', size=18, color='gold', line=dict(color='black', width=1.5)),
    name='Theory (0.5, 2/3)', showlegend=True))

# Trajectory line
fig.add_trace(go.Scatter(
    x=traj_betas, y=traj_relax, mode='lines',
    line=dict(color='rgba(0,0,0,0.3)', width=1.5, dash='dot'),
    showlegend=False))

# Points with step labels
for i, (label, r) in enumerate(R.items()):
    rdf = r['relax_df_fixed']
    if rdf.empty:
        continue
    beta_v = r['beta']['beta']
    relax_v = rdf['ratio_u3'].median()
    fig.add_trace(go.Scatter(
        x=[beta_v], y=[relax_v], mode='markers+text',
        marker=dict(size=14, color=STEP_COLORS[label],
                    line=dict(color='black', width=1.5)),
        text=[f"{r['step']//1000}K"], textposition='top center',
        textfont=dict(size=10),
        name=label, showlegend=True))

# Arrows between consecutive points
for i in range(len(traj_betas) - 1):
    fig.add_annotation(
        x=traj_betas[i+1], y=traj_relax[i+1],
        ax=traj_betas[i], ay=traj_relax[i],
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=3, arrowsize=1.5, arrowwidth=1.5,
        arrowcolor='rgba(0,0,0,0.4)')

pub_layout(fig, width=SINGLE_W, height=SINGLE_W, legend_pos='bl')
fig.update_xaxes(title_text='beta')
fig.update_yaxes(title_text='Relaxation ratio r@u=3')
save_fig(fig, 'beta_vs_relax_trajectory.png')
fig.show()

  Saved: beta_vs_relax_trajectory.png


In [26]:
# -- KEY FIGURE: Multi-Metric Dashboard 2x2 (beta, relax, gamma, stability vs step) --

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['<b>Beta (square-root law)</b>',
                    '<b>Relaxation ratio (r@u=3)</b>',
                    '<b>Decay exponent (gamma)</b>',
                    '<b>Stability fraction</b>'],
    horizontal_spacing=0.12, vertical_spacing=0.12)

steps_arr = np.array([r['step'] for r in R.values()])
marker_colors = [STEP_COLORS[l] for l in R.keys()]

# (1,1) Beta
beta_vals = [r['beta']['beta'] for r in R.values()]
fig.add_trace(go.Scatter(
    x=steps_arr, y=beta_vals, mode='lines+markers',
    line=dict(color='#1f78b4', width=2),
    marker=dict(size=9, color=marker_colors, line=dict(color='black', width=1)),
    showlegend=False), row=1, col=1)
fig.add_hline(y=0.5, line_dash='dash', line_color='black', line_width=1, row=1, col=1)

# (1,2) Relaxation
relax_vals = [r['relax_df_fixed']['ratio_u3'].median() if not r['relax_df_fixed'].empty else np.nan
              for r in R.values()]
fig.add_trace(go.Scatter(
    x=steps_arr, y=relax_vals, mode='lines+markers',
    line=dict(color='#1f78b4', width=2),
    marker=dict(size=9, color=marker_colors, line=dict(color='black', width=1)),
    showlegend=False), row=1, col=2)
fig.add_hline(y=2/3, line_dash='dash', line_color='black', line_width=1, row=1, col=2)

# (2,1) Gamma
gamma_vals = [decay_results[l]['gamma'].median() if not decay_results[l].empty else np.nan
              for l in R.keys()]
fig.add_trace(go.Scatter(
    x=steps_arr, y=gamma_vals, mode='lines+markers',
    line=dict(color='#1f78b4', width=2),
    marker=dict(size=9, color=marker_colors, line=dict(color='black', width=1)),
    showlegend=False), row=2, col=1)
fig.add_hline(y=0.5, line_dash='dot', line_color='grey', line_width=1, row=2, col=1)
fig.add_hline(y=0.8, line_dash='dot', line_color='grey', line_width=1, row=2, col=1)

# (2,2) Stability
stab_vals = [r['stab_df']['stabilized'].mean() if not r['stab_df'].empty else 0
             for r in R.values()]
fig.add_trace(go.Scatter(
    x=steps_arr, y=stab_vals, mode='lines+markers',
    line=dict(color='#1f78b4', width=2),
    marker=dict(size=9, color=marker_colors, line=dict(color='black', width=1)),
    showlegend=False), row=2, col=2)

# Axis labels
fig.update_xaxes(title_text='Step', row=2, col=1)
fig.update_xaxes(title_text='Step', row=2, col=2)
fig.update_yaxes(title_text='beta', row=1, col=1)
fig.update_yaxes(title_text='r@u=3', row=1, col=2)
fig.update_yaxes(title_text='gamma', row=2, col=1)
fig.update_yaxes(title_text='Frac. stable', row=2, col=2)

fig.update_layout(
    width=FULL_W, height=int(FULL_W * 0.7),
    template='plotly_white',
    font=dict(family='Times New Roman, DejaVu Serif, serif', size=12, color='black'),
    margin=dict(l=55, r=15, t=35, b=55),
)
fig.update_xaxes(**_AX)
fig.update_yaxes(**_AX)
fig.update_annotations(font_size=13)
save_fig(fig, 'dashboard_4metrics_vs_step.png')
fig.show()

  Saved: dashboard_4metrics_vs_step.png


---
## Part 6: No-Arbitrage Consistency

Do checkpoints progressively satisfy more no-arbitrage conditions?

In [27]:
# -- Permanent / Temporary Decomposition per checkpoint --

decomp_results = OrderedDict()

for label, r_data in R.items():
    rdf = r_data['relax_df'].copy()
    if rdf.empty or len(rdf) < 3:
        continue

    rdf['I_perm'] = rdf['I_final']
    rdf['I_temp'] = rdf['I_peak'] - rdf['I_final']

    pc = r_data['pc']

    folder_vday = {}
    for f in rdf['folder'].unique():
        pc_f = pc[pc['folder'] == f]
        if not pc_f.empty:
            i_val, c_val, mb_val, V_val = parse_folder_params_v2(f)
            Q_total = i_val * V_val
            last_ins = pc_f[pc_f['insertion_idx'] == pc_f['insertion_idx'].max()]
            if not last_ins.empty:
                med_x = last_ins['x'].median()
                folder_vday[f] = Q_total / np.exp(med_x)

    folder_sigma = {}
    for f in rdf['folder'].unique():
        pc_f = pc[pc['folder'] == f]
        if not pc_f.empty:
            folder_sigma[f] = np.exp(pc_f['alpha'].median())

    valid = []
    for _, row in rdf.iterrows():
        f = row['folder']
        if f not in folder_vday or f not in folder_sigma:
            continue
        V_day = folder_vday[f]
        sigma = folder_sigma[f]
        QV = row['Q'] / V_day
        if QV <= 0 or sigma <= 0:
            continue
        valid.append({
            'folder': f, 'Q': row['Q'], 'V': row['V'],
            'QV': QV, 'sigma': sigma,
            'I_perm': row['I_perm'], 'I_temp': row['I_temp'],
            'I_peak': row['I_peak'],
            'ln_QV': np.log(QV),
            'ln_Iperm_s': np.log(abs(row['I_perm']) / sigma + 1e-30),
            'ln_Itemp_s': np.log(abs(row['I_temp']) / sigma + 1e-30),
        })
    vdf = pd.DataFrame(valid)
    if len(vdf) < 3:
        continue

    ok_p = np.isfinite(vdf['ln_QV']) & np.isfinite(vdf['ln_Iperm_s']) & (vdf['I_perm'] > 0)
    if ok_p.sum() >= 3:
        sl_p = linregress(vdf.loc[ok_p, 'ln_QV'], vdf.loc[ok_p, 'ln_Iperm_s'])
        beta_perm = sl_p.slope
        r2_perm = sl_p.rvalue**2
    else:
        beta_perm, r2_perm = np.nan, np.nan

    ok_t = np.isfinite(vdf['ln_QV']) & np.isfinite(vdf['ln_Itemp_s']) & (vdf['I_temp'] > 0)
    if ok_t.sum() >= 3:
        sl_t = linregress(vdf.loc[ok_t, 'ln_QV'], vdf.loc[ok_t, 'ln_Itemp_s'])
        beta_temp = sl_t.slope
        r2_temp = sl_t.rvalue**2
    else:
        beta_temp, r2_temp = np.nan, np.nan

    decomp_results[label] = {
        'beta_perm': beta_perm, 'r2_perm': r2_perm,
        'beta_temp': beta_temp, 'r2_temp': r2_temp,
        'vdf': vdf,
    }

print("\n-- Permanent/Temporary Decomposition --")
print("  Theory: beta_perm ~ 1.0 (Huberman-Stanzl), beta_temp ~ 0.5")
for label, dr in decomp_results.items():
    print(f"  {label:15s}  beta_perm={dr['beta_perm']:.3f} (R2={dr['r2_perm']:.3f})  "
          f"beta_temp={dr['beta_temp']:.3f} (R2={dr['r2_temp']:.3f})")


-- Permanent/Temporary Decomposition --
  Theory: beta_perm ~ 1.0 (Huberman-Stanzl), beta_temp ~ 0.5
  Step 22820       beta_perm=1.179 (R2=0.317)  beta_temp=-0.236 (R2=0.011)
  Step 38335       beta_perm=1.231 (R2=0.309)  beta_temp=0.005 (R2=0.000)
  Step 79688       beta_perm=0.901 (R2=0.341)  beta_temp=0.205 (R2=0.031)
  Step 100378      beta_perm=0.587 (R2=0.159)  beta_temp=0.228 (R2=0.010)


In [28]:
# -- No-Arbitrage Consistency Scorecard (5 tests x 4 checkpoints) --

arb_rows = []

for label, r_data in R.items():
    delta = r_data['beta']['beta']
    relax_med = r_data['relax_df']['ratio'].median() if not r_data['relax_df'].empty else np.nan

    ddf = decay_results.get(label, pd.DataFrame())
    gamma_med = ddf['gamma'].median() if not ddf.empty else np.nan

    dr = decomp_results.get(label, {})
    beta_perm = dr.get('beta_perm', np.nan)

    test_A = delta < 1.0 if np.isfinite(delta) else False
    test_B = (0.7 <= beta_perm <= 1.3) if np.isfinite(beta_perm) else False
    test_C = (0.3 <= gamma_med <= 1.0) if np.isfinite(gamma_med) else False
    test_D = (0.5 <= relax_med <= 1.0) if np.isfinite(relax_med) else False
    if np.isfinite(delta) and np.isfinite(gamma_med) and gamma_med > 0:
        gatheral_bound = 1.0 / (1.0 + 2.0 * gamma_med)
        test_E = delta <= gatheral_bound
    else:
        gatheral_bound = np.nan
        test_E = False

    n_pass = sum([test_A, test_B, test_C, test_D, test_E])

    arb_rows.append({
        'Checkpoint': label,
        'Step': r_data['step'],
        'delta': f'{delta:.3f}',
        'beta_perm': f'{beta_perm:.3f}' if np.isfinite(beta_perm) else '---',
        'gamma': f'{gamma_med:.3f}' if np.isfinite(gamma_med) else '---',
        'relax_r': f'{relax_med:.3f}' if np.isfinite(relax_med) else '---',
        'A:Concav': 'PASS' if test_A else 'FAIL',
        'B:Perm~1': 'PASS' if test_B else 'FAIL',
        'C:Decay': 'PASS' if test_C else 'FAIL',
        'D:Relax': 'PASS' if test_D else 'FAIL',
        'E:Gather': 'PASS' if test_E else 'FAIL',
        'Score': f'{n_pass}/5',
        'n_pass': n_pass,
    })

arb_table = pd.DataFrame(arb_rows)
print('\n-- No-Arbitrage Consistency Check (5 tests x 4 checkpoints) --')
print(arb_table.drop(columns=['n_pass']).to_string(index=False))


-- No-Arbitrage Consistency Check (5 tests x 4 checkpoints) --
 Checkpoint   Step delta beta_perm gamma relax_r A:Concav B:Perm~1 C:Decay D:Relax E:Gather Score
 Step 22820  22820 0.526     1.179 0.482   0.655     PASS     PASS    PASS    PASS     FAIL   4/5
 Step 38335  38335 0.526     1.231 0.393   0.826     PASS     PASS    PASS    PASS     PASS   5/5
 Step 79688  79688 0.555     0.901 0.283   0.704     PASS     PASS    FAIL    PASS     PASS   4/5
Step 100378 100378 0.546     0.587 0.344   0.877     PASS     FAIL    PASS    PASS     PASS   4/5


In [29]:
# -- KEY FIGURE: No-Arb Score vs Training Step (bar) --

fig = go.Figure(go.Bar(
    x=[r['Checkpoint'] for r in arb_rows],
    y=[r['n_pass'] for r in arb_rows],
    marker_color=[STEP_COLORS[r['Checkpoint']] for r in arb_rows],
    width=0.55,
    marker_line_color='black', marker_line_width=1,
    text=[r['Score'] for r in arb_rows],
    textposition='outside'))

pub_layout(fig, width=FULL_W, height=400, legend_pos='none',
           yaxis_range=[0, 5.5])
fig.update_xaxes(title_text='', tickangle=-30)
fig.update_yaxes(title_text='No-Arbitrage Score (out of 5)', dtick=1)
save_fig(fig, 'noarb_score_vs_step.png')
fig.show()

  Saved: noarb_score_vs_step.png


In [30]:
# -- Figure: No-Arb Scatter with trajectory (beta_perm vs relaxation) --

fig = go.Figure()

# Acceptable zone
fig.add_shape(type='rect', x0=0.7, x1=1.3, y0=0.5, y1=1.0,
              fillcolor='rgba(0,200,0,0.08)', line=dict(color='green', width=1, dash='dot'))

# Theory point
fig.add_trace(go.Scatter(
    x=[1.0], y=[2/3], mode='markers',
    marker=dict(symbol='star', size=18, color='gold', line=dict(color='black', width=1.5)),
    name='Theory (1.0, 2/3)', showlegend=True))

# Collect trajectory
traj_bp = []
traj_rr = []
for label in R:
    dr = decomp_results.get(label, {})
    bp = dr.get('beta_perm', np.nan)
    rdf = R[label]['relax_df']
    relax_med = rdf['ratio'].median() if not rdf.empty else np.nan
    traj_bp.append(bp)
    traj_rr.append(relax_med)

# Trajectory line
valid_traj = [(bp, rr) for bp, rr in zip(traj_bp, traj_rr) if np.isfinite(bp) and np.isfinite(rr)]
if len(valid_traj) > 1:
    fig.add_trace(go.Scatter(
        x=[v[0] for v in valid_traj], y=[v[1] for v in valid_traj],
        mode='lines', line=dict(color='rgba(0,0,0,0.3)', width=1.5, dash='dot'),
        showlegend=False))

# Points
for i, label in enumerate(R.keys()):
    bp, rr = traj_bp[i], traj_rr[i]
    if not np.isfinite(bp) or not np.isfinite(rr):
        continue
    fig.add_trace(go.Scatter(
        x=[bp], y=[rr], mode='markers+text',
        marker=dict(size=12, color=STEP_COLORS[label],
                    line=dict(color='black', width=1)),
        text=[f"{R[label]['step']//1000}K"], textposition='top center',
        textfont=dict(size=10),
        name=label, showlegend=True))

# Arrows
for i in range(len(valid_traj) - 1):
    fig.add_annotation(
        x=valid_traj[i+1][0], y=valid_traj[i+1][1],
        ax=valid_traj[i][0], ay=valid_traj[i][1],
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowhead=3, arrowsize=1.5, arrowwidth=1.5,
        arrowcolor='rgba(0,0,0,0.4)')

pub_layout(fig, width=SINGLE_W, height=SINGLE_W, legend_pos='bl')
fig.update_xaxes(title_text='beta_perm')
fig.update_yaxes(title_text='Relaxation ratio r')
save_fig(fig, 'noarb_scatter_trajectory.png')
fig.show()

  Saved: noarb_scatter_trajectory.png


---
## Part 7: Summary

In [31]:
# -- Summary: Deltas (first -> last checkpoint) --

labels_list = list(R.keys())
first_label = labels_list[0]
last_label  = labels_list[-1]
first_r = R[first_label]
last_r  = R[last_label]

print(f"{'='*70}")
print(f"  CHECKPOINT EVOLUTION SUMMARY: {first_label} -> {last_label}")
print(f"{'='*70}")
print()

# Beta
b_first = first_r['beta']['beta']
b_last  = last_r['beta']['beta']
print(f"  Beta (global):     {b_first:.3f} -> {b_last:.3f}  (delta={b_last - b_first:+.3f}, theory=0.500)")
bs_first = first_r['strat']['beta_stratified']
bs_last  = last_r['strat']['beta_stratified']
print(f"  Beta (stratified): {bs_first:.3f} -> {bs_last:.3f}  (delta={bs_last - bs_first:+.3f})")

# Relaxation
r_first = first_r['relax_df_fixed']['ratio_u3'].median() if not first_r['relax_df_fixed'].empty else np.nan
r_last  = last_r['relax_df_fixed']['ratio_u3'].median() if not last_r['relax_df_fixed'].empty else np.nan
print(f"  Relax@u=3:         {r_first:.3f} -> {r_last:.3f}  (delta={r_last - r_first:+.3f}, theory=0.667)")

# Gamma
ddf_first = decay_results.get(first_label, pd.DataFrame())
ddf_last  = decay_results.get(last_label, pd.DataFrame())
g_first = ddf_first['gamma'].median() if not ddf_first.empty else np.nan
g_last  = ddf_last['gamma'].median() if not ddf_last.empty else np.nan
print(f"  Gamma (decay):     {g_first:.3f} -> {g_last:.3f}  (delta={g_last - g_first:+.3f}, target=[0.5,0.8])")

# Stability
s_first = first_r['stab_df']['stabilized'].mean() if not first_r['stab_df'].empty else 0
s_last  = last_r['stab_df']['stabilized'].mean() if not last_r['stab_df'].empty else 0
print(f"  Stability:         {s_first:.0%} -> {s_last:.0%}  (delta={s_last - s_first:+.0%})")

# No-arb score
arb_first = next((r for r in arb_rows if r['Checkpoint'] == first_label), None)
arb_last  = next((r for r in arb_rows if r['Checkpoint'] == last_label), None)
if arb_first and arb_last:
    print(f"  No-Arb Score:      {arb_first['Score']} -> {arb_last['Score']}")

print()
print(f"{'='*70}")

  CHECKPOINT EVOLUTION SUMMARY: Step 22820 -> Step 100378

  Beta (global):     0.526 -> 0.546  (delta=+0.020, theory=0.500)
  Beta (stratified): 0.513 -> 0.541  (delta=+0.028)
  Relax@u=3:         0.868 -> 0.842  (delta=-0.026, theory=0.667)
  Gamma (decay):     0.482 -> 0.344  (delta=-0.138, target=[0.5,0.8])
  Stability:         0% -> 0%  (delta=+0%)
  No-Arb Score:      4/5 -> 4/5



## Conclusions

**Architecture**: S5-4K (j2504167), 55M parameters, 24-token encoding, 4K context length.

**Training steps analyzed**: 22,820 / 38,335 / 79,688 / 100,378

**Key questions answered**:

1. **Beta convergence**: Does the square-root law exponent approach the theoretical value of 0.5 with more training?
2. **Decay quality**: Do relaxation and decay exponents enter physically meaningful ranges?
3. **Stability**: Does the fraction of stable impact curves increase?
4. **No-arbitrage**: Does the model progressively satisfy more no-arbitrage conditions?

Since all checkpoints share the same 55M architecture, any observed improvements are purely due to **better learning** (training dynamics), not increased model capacity. This cleanly separates the "learning" axis from the "capacity" axis in the 7-model comparison (NB 201).